# Spectrum amplification for quantum chemistry with SOSSA

## Why the Hamiltonian representation decides the cost

The electronic-structure Hamiltonian in second quantization is

$$
H=\sum_{pq}h_{pq}E_{pq}
+\frac{1}{2}\sum_{pqrs}g_{pqrs}E_{pq}E_{rs}
+E_{\mathrm{nuc}},
\qquad
E_{pq}=\sum_{\sigma}a_{p\sigma}^{\dagger}a_{q\sigma}.
$$

Every fault-tolerant ground-state algorithm inherits its cost from how this operator is
*represented*. Qubitization block-encodes $H/\lambda$ and runs phase estimation on the resulting
walk operator, and the number of walk queries needed to reach an energy precision $\sigma_E$ is

$$
p=\left\lceil\frac{\pi\lambda}{2\sigma_E}\right\rceil .
$$

So the block-encoding 1-norm $\lambda$ — a property of the *representation*, not of the molecule —
sets the runtime. The [stretched N<sub>2</sub> QPE notebook](qpe_stretched_n2.ipynb) builds exactly
this pipeline with a linear-combination-of-unitaries (LCU) block encoding, alongside a
Trotter-based alternative, and is the recommended prerequisite to this one.

**Sum-of-squares spectral amplification (SOSSA)** attacks $\lambda$ directly. Suppose the
Hamiltonian can be written as a sum of squares,

$$
H=\sum_{\alpha}O_{\alpha}^{\dagger}O_{\alpha}+E_{\mathrm{SOS}}
=H_{\mathrm{sqrt}}^{\dagger}H_{\mathrm{sqrt}}+E_{\mathrm{SOS}} .
$$

Then $H-E_{\mathrm{SOS}}$ is positive semidefinite, and block encoding the *square root*
$H_{\mathrm{sqrt}}$ instead of $H$ stretches the low-energy edge of the spectrum. Near the ground
state the query count is governed not by $\Lambda=\tfrac12\sum_{\alpha}\lambda_{\alpha}^{2}$ but by

$$
\lambda_{\mathrm{eff}}=\sqrt{E_{\mathrm{gap}}\left(2\Lambda-E_{\mathrm{gap}}\right)},
\qquad
E_{\mathrm{gap}}=E_{\mathrm{gs}}-E_{\mathrm{SOS}} .
$$

Because the ground state sits near the bottom of the representable window $[0,2\Lambda]$, we have
$E_{\mathrm{gap}}\ll 2\Lambda$ and therefore $\lambda_{\mathrm{eff}}\ll\Lambda$: the same energy
precision costs fewer queries. That is the entire point of this notebook, and everything below
exists to turn a chemistry Hamiltonian into a form where it applies.

This notebook follows Low *et al.*, *Fast quantum simulation of electronic structure by spectrum
amplification*, [arXiv:2502.15882](https://arxiv.org/abs/2502.15882).

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md),
install the `jupyter`, `plugins`, and `qre` extras:

```bash
pip install 'qdk-chemistry[jupyter,plugins,qre]'
```

## Getting a chemistry Hamiltonian into sum-of-squares form: DFTHC

The one-body term of $H$ is already a quadratic form and causes no trouble. The obstacle is the
two-electron tensor $g_{pqrs}$, which has $O(N^4)$ entries and is not a sum of squares as written.

**Double-factorized tensor hypercontraction (DFTHC)** rewrites it as one. It replaces $g_{pqrs}$
with three small tensors — basis vectors $u_{b,p}^{(r)}$, weights $w_{b}^{(rc)}$ and identity
weights $W^{(rc)}$ — indexed by a rank $r\in[R]$, a basis $b\in[B]$ and a copy $c\in[C]$:

$$
\begin{aligned}
H_{\mathrm{DFTHC}}
={}&\sum_{pq}h_{pq}^{(1)\prime}E_{pq}
+\frac{1}{2}\sum_{r\in[R]}\sum_{c\in[C]}\left(
W^{(rc)}\mathbb{I}
+\sum_{b=0}^{B-1}w_b^{(rc)}\sum_{pq}u_{b,p}^{(r)}u_{b,q}^{(r)}E_{pq}
\right)^2 \\
&-\frac{1}{2}\sum_{r\in[R]}\sum_{c\in[C]}\left|W^{(rc)}\right|^2 .
\end{aligned}
$$

Two things to notice.

* The two-body term is now **literally a square**, which is what makes the sum-of-squares form —
  and therefore spectrum amplification — available at all.
* The one-body matrix is $h^{(1)\prime}$, not the original $h_{pq}$. Squaring the two-body
  generators produces normal-ordering corrections that are folded back into the one-body part, and
  the leftover constant becomes part of $E_{\mathrm{SOS}}$. A DFTHC container is therefore *not*
  interchangeable with the raw integrals; it is a different, self-consistent set of tensors.

The storage drops from $O(N^4)$ to $O(RBN)$, but the factors do not come from a closed-form
decomposition the way a Cholesky factorization does. Fitting $(u, w, W)$ so that
$H_{\mathrm{DFTHC}}\approx H$ within chemical accuracy is a **hard non-convex numerical
optimization**, and the published $(N,R,B,C)$ shapes are the output of that optimization. This
notebook *consumes* a converged DFTHC container; producing one is out of scope.

### Double factorization is the easy special case

Conventional double factorization (DF) is DFTHC with $R=R_{\mathrm{DF}}$, $B=N$ and $C=1$, in which
each two-body factor is simply diagonalized:

$$
g_{pqrs}\approx\sum_{Q=0}^{R_{\mathrm{DF}}-1}L_{pq}^{(Q)}L_{rs}^{(Q)},
\qquad
L_{pq}^{(Q)}=\sum_{b=0}^{N-1}w_b^{(Q)}u_{b,p}^{(Q)}u_{b,q}^{(Q)} .
$$

Because the $u^{(Q)}$ are eigenvectors, no optimization is needed — an eigendecomposition suffices.
DF is therefore the cheap, always-available way to reach sum-of-squares form, at the price of
larger $(B, C)$ than a tuned DFTHC fit would need. `qdk-chemistry` ships it as the
`double_factorization` algorithm, and its output feeds the same SOSSA pipeline.

## Choose a run mode

The rest of the notebook is a **single pass** through the SOSSA pipeline. Only the first step —
where the factorized Hamiltonian comes from — depends on what you want out of it, so pick one mode
now and the remaining cells follow unchanged.

| `RUN_MODE` | Hamiltonian | What you get |
|---|---|---|
| `"stored"` | A pre-fitted DFTHC H<sub>2</sub> shipped with the examples | The full pipeline **validated end to end** against a state-vector simulation, plus a resource estimate |
| `"double_factorization"` | Stretched N<sub>2</sub>, double-factorized from scratch | A realistic classical-to-quantum workflow, ending in a resource estimate |
| `"synthetic"` | Random tensors with a chosen $(N,R,B,C)$ | Resource estimation and trade-off exploration **only** |

A note on the synthetic mode: only the *dimensions* of those tensors are physical. They drive
circuit size, and therefore cost, but the spectrum is meaningless. Energies must never be read off
a synthetic run, and the notebook refuses to simulate one.

Simulation is also guarded by size. A state-vector simulator is exponential in qubit count, so a
50-orbital active space — tens of thousands of logical qubits once the walk ancillas are included —
must never be handed to one. `MAX_SIMULATION_QUBITS` below sets the cutoff, and the validation step
is skipped with a printed reason whenever the estimate exceeds it.

In [ ]:
from pathlib import Path

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import AlgorithmRef, Hamiltonian, MajoranaMapping
from qdk_chemistry.utils import Logger
from utils.sossa_utils import (
    SOSSA_QPE_BUILDER,
    build_sossa_qpe_circuit,
    compiled_circuit_mapper,
    hartree_fock_state_preparation,
    heisenberg_queries,
    make_fake_hamiltonian,
    require_sossa_walk,
    simulation_qubit_estimate,
)

Logger.set_global_level(Logger.LogLevel.off)

# Pick one: "stored", "double_factorization" or "synthetic".
RUN_MODE = "stored"

RUN_MODES = ("stored", "double_factorization", "synthetic")
if RUN_MODE not in RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {RUN_MODES}, got {RUN_MODE!r}.")

# Target energy precision, and the resource estimator's total error budget.
TARGET_PRECISION = 1e-3
MAX_ERROR = 0.01

# Validation guard. A state-vector simulator is exponential in qubit count, so anything
# wider than this is resource-estimated but never simulated.
MAX_SIMULATION_QUBITS = 26
SIMULATION_QUERIES = 7
SIMULATION_SHOTS = 50
SIMULATION_SEED = 42

# Published (N, R, B, C) shapes and normalizations for the synthetic mode.
SYNTHETIC_SYSTEMS = {
    "Fe2S2 (30e, 20o)": {
        "electrons": 30,
        "N": 20,
        "R": 14,
        "B": 15,
        "C": 5,
        "lambda_eff": 6.4690,
        "b_coeff": 11,
        "b_rot": 15,
    },
    "FeMoCo (54e, 54o)": {
        "electrons": 54,
        "N": 54,
        "R": 10,
        "B": 27,
        "C": 27,
        "lambda_eff": 21.3674,
        "b_coeff": 9,
        "b_rot": 16,
    },
}
SYNTHETIC_SYSTEM = "Fe2S2 (30e, 20o)"

# Defaults the run modes below may override.
reference_energy = None  # Classical E_gs, used to derive lambda_eff.
lambda_eff_override = None  # Published lambda_eff, used by the synthetic mode.
rotation_bits = 15
coefficient_bits = 11

## Step 1: obtain a factorized Hamiltonian

Each of the three cells below is inert unless its mode is selected. Exactly one of them defines
`hamiltonian`, the electron counts of the reference determinant, and — where a classical reference
energy exists — the `reference_energy` that Step 3 turns into $\lambda_{\mathrm{eff}}$.

### Mode 1: a stored DFTHC Hamiltonian

The examples ship a converged DFTHC fit for H<sub>2</sub> with $N=2$ spatial orbitals, $R=2$ ranks,
$B=2$ bases and $C=1$ copy. It is small enough to simulate, which makes it the only mode that can
check the whole pipeline against an actual measured energy.

In [ ]:
if RUN_MODE == "stored":
    json_path = Path("data") / "h2_dfthc_r2_b2_c1.hamiltonian.json"
    hamiltonian = Hamiltonian.from_json(json_path.read_text())
    num_alpha, num_beta = 1, 1
    system_label = "H2 (stored DFTHC, N=2 R=2 B=2 C=1)"

### Mode 2: double factorization from scratch

This mode runs the classical workflow — SCF, orbital localization, active-space selection, CASCI —
and then double-factorizes the active-space Hamiltonian. The classical part is condensed here; see
the [stretched N<sub>2</sub> QPE notebook](qpe_stretched_n2.ipynb) for the full discussion of each
step.

The CASCI energy is kept as `reference_energy`. It is what makes $\lambda_{\mathrm{eff}}$
computable, and therefore what lets this mode show the spectrum-amplification saving rather than a
conservative bound.

In [ ]:
if RUN_MODE == "double_factorization":
    from qdk_chemistry.data import Structure
    from qdk_chemistry.data.symmetry import SymmetryLabel, axes
    from qdk_chemistry.utils import compute_valence_space_parameters

    structure = Structure.from_xyz_file(Path("data/stretched_n2.structure.xyz"))
    e_hf, wfn_hf = create("scf_solver").run(
        structure,
        charge=0,
        spin_multiplicity=1,
        basis_or_guess="cc-pvdz",
    )

    num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
    valence_wfn = create(
        "active_space_selector",
        "qdk_valence",
        num_active_electrons=num_val_e,
        num_active_orbitals=num_val_o,
    ).run(wfn_hf)

    valence_indices = valence_wfn.get_orbitals().active_indices()
    localized_wfn = create("orbital_localizer", "qdk_mp2_natural_orbitals").run(
        valence_wfn,
        list(valence_indices.indices(SymmetryLabel([axes.alpha()]))),
        list(valence_indices.indices(SymmetryLabel([axes.beta()]))),
    )

    hamiltonian_constructor = create("hamiltonian_constructor")
    localized_hamiltonian = hamiltonian_constructor.run(localized_wfn.get_orbitals())
    num_alpha_electrons, num_beta_electrons = localized_wfn.get_active_num_electrons()
    _, selected_ci_wfn = create(
        "multi_configuration_calculator",
        "macis_asci",
        calculate_one_rdm=True,
        calculate_two_rdm=True,
    ).run(localized_hamiltonian, num_alpha_electrons, num_beta_electrons)

    autocas_wfn = create("active_space_selector", "qdk_autocas_eos").run(
        selected_ci_wfn
    )
    active_hamiltonian = hamiltonian_constructor.run(autocas_wfn.get_orbitals())
    num_alpha, num_beta = autocas_wfn.get_active_num_electrons()

    e_cas, _ = create("multi_configuration_calculator", "macis_cas").run(
        active_hamiltonian, num_alpha, num_beta
    )
    print(f"Hartree-Fock energy: {e_hf:.6f} Hartree")
    print(f"Active-space CASCI energy: {e_cas:.6f} Hartree")
    print(f"Active space: {num_alpha} alpha + {num_beta} beta electrons")

    factorizer = create("hamiltonian_factorization", "double_factorization")
    factorizer.settings().set("truncation_threshold", 1e-8)
    hamiltonian = factorizer.run(active_hamiltonian)

    reference_energy = e_cas
    system_label = "stretched N2 (double factorization)"

### Mode 3: a synthetic Hamiltonian of arbitrary size

This mode fabricates tensors with the $(N,R,B,C)$ shape of a published active space so that
circuits of a realistic size can be built and costed. The entries are random: the operator is *not*
the molecule's Hamiltonian and its spectrum means nothing. Only structural quantities — register
widths, oracle sizes, query counts — carry over, which is exactly what a resource estimate consumes.

Because the tensors carry no spectrum, `reference_energy` stays `None` and the published
$\lambda_{\mathrm{eff}}$ is supplied directly instead.

In [ ]:
if RUN_MODE == "synthetic":
    params = SYNTHETIC_SYSTEMS[SYNTHETIC_SYSTEM]
    hamiltonian = make_fake_hamiltonian(
        params["N"], params["R"], params["B"], params["C"]
    )
    num_alpha = (params["electrons"] + 1) // 2
    num_beta = params["electrons"] // 2
    lambda_eff_override = params["lambda_eff"]
    rotation_bits = params["b_rot"]
    coefficient_bits = params["b_coeff"]
    system_label = f"synthetic {SYNTHETIC_SYSTEM}"

In [ ]:
print(f"Run mode: {RUN_MODE}")
print(f"System:   {system_label}")
print(hamiltonian.get_summary())

## Step 2: map the factorization to a sum-of-squares qubit operator

The `sum_of_squares` qubit mapper applies the Jordan-Wigner transformation to the factorized
Hamiltonian and emits the generators $O_\alpha$ explicitly:

$$
\begin{aligned}
H\approx H_{\mathrm{DFTHC}}
={}&\sum_{G\in\{\mathrm{D}_1,\mathrm{Q}_1\}}\sum_r\sum_{\sigma\in\{0,1\}}
O_{G^{\sigma,r}}^{\dagger}O_{G^{\sigma,r}} \\
&+\sum_{r\in[R],c\in[C]}O_{\mathrm{SF},rc}^{\dagger}O_{\mathrm{SF},rc}
+E_{\mathrm{SOS}} .
\end{aligned}
$$

There are two families. The **one-body** generators come from diagonalizing $h^{(1)\prime}$: a
positive eigenvalue yields a particle generator $\mathrm{D}_1$, a negative one a hole generator
$\mathrm{Q}_1$. The **spin-free** generators $O_{\mathrm{SF},rc}$ are the squared two-body factors,
one per $(r,c)$ pair. The constant $E_{\mathrm{SOS}}$ absorbs the normal-ordering leftovers and the
nuclear repulsion; it is added back when a phase is decoded into an energy.

In [ ]:
container = hamiltonian.get_container()
num_orbitals = container.get_num_orbitals()

operator = create("qubit_mapper", "sum_of_squares").run(
    hamiltonian,
    MajoranaMapping.jordan_wigner(2 * num_orbitals),
)
print(operator.get_container().get_summary())

## Step 3: build the SOSSA walk and size the query schedule

The `sossa` unitary builder turns the generators into a block encoding of $H_{\mathrm{sqrt}}$,

$$
U:=\mathrm{SEL}\cdot\mathrm{PREP}^{\dagger}
=\mathrm{BE}\!\left[\frac{H_{\mathrm{sqrt}}^{\prime}}{\lambda_{\mathrm{sqrt}}}\right],
\qquad
\lambda_{\mathrm{sqrt}}=\sqrt{\sum_{\alpha}\lambda_{\alpha}^{2}},
$$

and wraps it in the reflections that make a self-inverse walk operator
$W=\mathrm{Ref}_{a,\mathcal{B}}\cdot U^{\dagger}\cdot\mathrm{Ref}_{\mathcal{B}}\cdot U$ whose
eigenphases satisfy $e^{\pm i\arccos(E_k/\Lambda-1)}$. Equivalently,

$$
\frac{2H_{\mathrm{SA}}}{\lambda_{\mathrm{sqrt}}^{2}}-\mathbb{I}
=\langle 0|_{\mathrm a\mathcal B}U^{\dagger}\mathrm{REF}_{\mathcal B}
U|0\rangle_{\mathrm a\mathcal B},
$$

so QPE on $W$ reads out the amplified spectrum rather than the raw one.

$\lambda_{\mathrm{eff}}$ is what converts that into a query count, and it needs
$E_{\mathrm{gap}}=E_{\mathrm{gs}}-E_{\mathrm{SOS}}$ — a *classical* input the factorization itself
does not carry. The container refuses to invent one: reading `lambda_eff` without supplying a
reference energy raises rather than returning a silently optimistic number. When no reference is
available we substitute the raw $\Lambda$, which is strictly conservative and simply forfeits the
amplification saving.

In [ ]:
sossa_settings = (
    {}
    if reference_energy is None
    else {"reference_ground_state_energy": reference_energy}
)
sossa_unitary_builder = AlgorithmRef(
    "hamiltonian_unitary_builder", "sossa", **sossa_settings
)

walk = create("hamiltonian_unitary_builder", "sossa", **sossa_settings).run(operator)
walk_container = require_sossa_walk(walk)

if lambda_eff_override is not None:
    lambda_effective = lambda_eff_override
    lambda_source = "published value for this system"
elif walk_container.has_lambda_eff:
    lambda_effective = walk_container.lambda_eff
    lambda_source = "derived from the classical reference energy"
else:
    lambda_effective = walk_container.normalization
    lambda_source = (
        "no reference energy available, falling back to the conservative Lambda"
    )

num_queries = heisenberg_queries(lambda_effective, TARGET_PRECISION)

print(walk_container.get_summary())
print(f"Lambda              = {walk_container.normalization:.6f} Hartree")
print(f"lambda_eff          = {lambda_effective:.6f} Hartree ({lambda_source})")
print(f"Queries for {TARGET_PRECISION:.0e} Ha : {num_queries}")

## Step 4: how the block encoding is actually applied

This is the part worth understanding if you want to read the DFTHC papers. The qubitization walk in
the [stretched N<sub>2</sub> notebook](qpe_stretched_n2.ipynb) uses the familiar
$\mathrm{PREPARE}$–$\mathrm{SELECT}$–$\mathrm{PREPARE}^{\dagger}$ skeleton over a flat list of
Pauli terms. SOSSA uses the same skeleton, but because DFTHC coefficients are naturally indexed by
*(generator, basis)* rather than by a flat term index, $\mathrm{PREPARE}$ splits into two levels.

**Outer PREPARE** loads the generator index $x_o$ over the $X_o=N+RC$ generators, weighted by each
generator's one-norm:

$$
\mathrm{PREP}|0\rangle_{\mathrm a}
=\sum_{x_o}\frac{\lambda_{x_o}}{\sqrt{2\Lambda}}|x_o\rangle|\mathrm{garbage}_{x_o}\rangle,
\qquad
\lambda_{\mathrm{SF},rc}=\frac{1}{\sqrt 2}\Big(|W^{(rc)}|+\sum_b |w_b^{(rc)}|\Big),
$$

with $\lambda_{\mathrm{D}_1,r}=\sqrt{w_+^{(r)}}$ and $\lambda_{\mathrm{Q}_1,r}=\sqrt{w_-^{(r)}}$ for
the one-body generators.

**Inner PREPARE** is conditioned on $x_o$ and loads the basis index $b\in[0,B]$ within the selected
generator, with amplitudes $\mathrm{sign}(w_b)\sqrt{|w_b|}$ — the square root is there because
PREPARE amplitudes enter the block encoding squared, and the sign is consumed later by SELECT.
Column $b=B$ addresses the identity term $W^{(rc)}\mathbb{I}$. One-body generators use a delta row,
since they have no basis sum.

**SELECT** applies the chosen generator: Givens rotations carry the orbitals into the eigenbasis of
rank $r$, the Majorana or rotated-$Z$ operator is applied, and the rotations are undone. A small
*free-rider* register carries the flags SELECT needs unconditionally — whether the generator is
spin-free, the sign, and the rank index — so they do not have to be recomputed.

Each of these is a pluggable algorithm, which is what makes the cost model interesting. The
`compiled_circuit_mapper` helper selects alias sampling for both PREPAREs and a QROM with a shared
phase gradient for SELECT: these are the compilations a fault-tolerant cost model should see. The
`direct` alternatives used for validation later realize the *same* walk operator with textbook
oracles that a state-vector simulator can execute.

In [ ]:
circuit_mapper = compiled_circuit_mapper(
    rotation_bit_precision=rotation_bits,
    coefficient_bit_precision=coefficient_bits,
)
print(f"Rotation angle precision:   {rotation_bits} bits")
print(f"PREPARE amplitude precision: {coefficient_bits} bits")

## Step 5: prepare the reference state

Phase estimation needs a trial state with appreciable overlap with the ground state. We use the
canonical Hartree-Fock determinant, loaded with the sparse-isometry state preparation. For a
strongly correlated system a multi-configuration trial state would be a better starting point; the
[stretched N<sub>2</sub> notebook](qpe_stretched_n2.ipynb) covers that choice and its cost.

In [ ]:
state_preparation = hartree_fock_state_preparation(hamiltonian, num_alpha, num_beta)

## Step 6: assemble the unary-iteration QPE circuit

Unary-iteration QPE prepares a phase register over $p+1$ slots and applies $p$ walk queries,
selecting which interleaved reflection is omitted by unary iteration over that register. Unlike a
binary-power schedule, $p$ can be *any* positive integer and need not be rounded up to a power of
two — which matters here, because $\lceil\pi\lambda_{\mathrm{eff}}/2\sigma_E\rceil$ rarely lands on
one. The phase register is prepared in a cosine window to suppress spectral leakage.

**This is the only phase-estimation builder that accepts a SOSSA walk.** Iterative and standard QPE
drive a block encoding through a `controlled_circuit_mapper`, and no controlled SOSSA mapper
exists, so those builders cannot construct a SOSSA circuit at all. `require_sossa_walk` in Step 3
already confirmed the container type; the builder re-checks it and raises on a mismatch rather than
producing a circuit that would return a wrong answer.

In [ ]:
qpe_builder = create(
    "qpe_circuit_builder",
    SOSSA_QPE_BUILDER,
    num_queries=num_queries,
    circuit_mapper=circuit_mapper,
    unitary_builder=sossa_unitary_builder,
)
circuit = qpe_builder.run(
    state_preparation=state_preparation,
    qubit_hamiltonian=operator,
)[0]

print(f"Walk queries:   {num_queries}")
print(f"Logical qubits: {circuit.num_qubits}")

## Step 7: validate against a state-vector simulation

The Heisenberg-limited schedule above is far beyond what a simulator can execute, so validation uses
a short walk — same block encoding, same reference state, simulator-friendly oracle compilations —
and checks that the measured phase decodes to the expected ground-state energy through
$E=2\Lambda\cos^{2}(\pi\varphi)+E_{\mathrm{SOS}}$.

Two guards apply before anything is simulated, and both report why they fired:

1. **Synthetic Hamiltonians are never simulated.** Their tensors are random, so a decoded energy
   would be a meaningless number presented in Hartree — worse than no number at all.
2. **Oversized problems are never simulated.** The qubit count is estimated from the walk container
   alone, before any circuit is built, so a large active space is rejected rather than sent to an
   exponential-memory simulator.

Note that the decoded energy has only the resolution of this short schedule, which is far coarser
than `TARGET_PRECISION`; it checks correctness of the pipeline, not the precision the resource
estimate is sized for. `python/tests/test_phase_estimation_sossa.py` pins the same path against an
independent exact diagonalization of the H<sub>2</sub> Hamiltonian.

In [ ]:
# This cell takes ~1 minute to run when simulation is enabled.
estimated_qubits = simulation_qubit_estimate(walk_container, SIMULATION_QUERIES)

if RUN_MODE == "synthetic":
    print(
        "Skipping simulation: synthetic tensors have no physical spectrum, so a decoded "
        "energy would be meaningless."
    )
elif estimated_qubits > MAX_SIMULATION_QUBITS:
    print(
        f"Skipping simulation: the walk needs about {estimated_qubits} qubits, over the "
        f"MAX_SIMULATION_QUBITS = {MAX_SIMULATION_QUBITS} cutoff."
    )
else:
    from qdk.widgets import Histogram

    validation_mapper = AlgorithmRef(
        "circuit_mapper",
        "sossa",
        outer_prepare_algorithm=AlgorithmRef("state_prep", "dense_pure_state"),
        inner_prepare_algorithm="direct",
        select_algorithm="direct",
    )
    validation_settings = {
        "num_queries": SIMULATION_QUERIES,
        "circuit_mapper": validation_mapper,
        "unitary_builder": sossa_unitary_builder,
    }
    validation_builder = AlgorithmRef(
        "qpe_circuit_builder", SOSSA_QPE_BUILDER, **validation_settings
    )

    validation_circuit = create(
        "qpe_circuit_builder", SOSSA_QPE_BUILDER, **validation_settings
    ).run(
        state_preparation=state_preparation,
        qubit_hamiltonian=operator,
    )[0]
    print(
        f"Validation circuit: {validation_circuit.num_qubits} qubits, {SIMULATION_QUERIES} queries"
    )

    execution = create(
        "circuit_executor", "qdk_sparse_state_simulator", seed=SIMULATION_SEED
    ).run(validation_circuit, shots=SIMULATION_SHOTS)
    display(
        Histogram(
            bar_values={
                bitstring: count / execution.total_shots
                for bitstring, count in execution.bitstring_counts.items()
            }
        )
    )

    qpe = create("phase_estimation", SOSSA_QPE_BUILDER, shots=SIMULATION_SHOTS)
    qpe.settings().set("qpe_circuit_builder", validation_builder)
    qpe.settings().set(
        "circuit_executor",
        AlgorithmRef(
            "circuit_executor", "qdk_sparse_state_simulator", seed=SIMULATION_SEED
        ),
    )
    qpe_result = qpe.run(
        qubit_hamiltonian=operator, state_preparation=state_preparation
    )

    print(f"Measured phase fraction: {qpe_result.phase_fraction:.6f}")
    print(f"Decoded ground-state energy: {qpe_result.raw_energy:.6f} Hartree")
    print(f"Sign-branch candidates: {tuple(round(e, 6) for e in qpe_result.branching)}")

## Step 8: physical resource estimates

`qdk.qre` maps the logical circuit onto a fault-tolerant architecture and returns Pareto-optimal
trade-offs between physical qubit count and runtime. We use a Majorana-based architecture with a
$10^{-5}$ physical error rate, the `ThreeAux` code, and round-based magic-state factories, with the
total error budget set by `MAX_ERROR`.

In [ ]:
from qdk.qre import estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

architecture = Majorana(error_rate=1e-5)
isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())

results = estimate(
    circuit.get_qre_application(),
    architecture,
    isa_query,
    max_error=MAX_ERROR,
    name=f"SOSSA {system_label}",
)
results.add_factory_summary_column()
display(results.as_frame())

plot_estimates(results, figsize=(6, 4))

## Limitations

* **SOSSA runs under unary-iteration QPE only.** The `sossa` circuit mapper emits the block encoding
  uncontrolled, which is what `qdk_unary` needs; iterative and standard QPE require a
  `controlled_circuit_mapper`, and none accepts a SOSSA walk. Both the helper used above and the
  builder itself check the container type and raise, so there is no path that quietly returns a
  wrong energy.
* **$\lambda_{\mathrm{eff}}$ needs a classical reference energy.** Without one, the container leaves
  it unset and raises on access rather than guessing. Falling back to $\Lambda$, as the `stored`
  mode does, over-estimates the query count instead of under-estimating it.
* **Synthetic Hamiltonians have no spectrum.** They exist to size circuits. Any energy decoded from
  one is noise, which is why simulation is refused in that mode.
* **The validation schedule is short.** It confirms the pipeline decodes correctly; it does not
  reach `TARGET_PRECISION`.
* **DFTHC fitting is out of scope.** This notebook consumes converged factors. Double factorization
  is available end to end because it needs only an eigendecomposition.

## Optional: trade-off exploration

With the pipeline understood, the synthetic mode can sweep it across systems to compare costs. The
cell below reuses `build_sossa_qpe_circuit`, which performs exactly the steps walked through above
in a single call, and switches to an automatic memory/compute architecture: compute qubits are
capped at 20% of the total, and the remaining qubits hold state without supporting gates, which
lets them use more efficient codes such as yoked surface codes.

In [ ]:
# This cell takes ~2 minutes to run.
if RUN_MODE != "synthetic":
    print(
        "The trade-off sweep runs in the 'synthetic' mode only; set RUN_MODE = 'synthetic'."
    )
else:
    from qdk.qre import PSSPC, DynamicMemoryCompute, LatticeSurgery

    MEMORY_COMPUTE_PERCENTAGE = 0.2
    trace_query = (
        DynamicMemoryCompute.q(compute_capacity_percentage=MEMORY_COMPUTE_PERCENTAGE)
        * PSSPC.q()
        * LatticeSurgery.q()
    )

    sweep_results = []
    for name, sweep_params in SYNTHETIC_SYSTEMS.items():
        sweep_circuit, _ = build_sossa_qpe_circuit(
            make_fake_hamiltonian(
                sweep_params["N"],
                sweep_params["R"],
                sweep_params["B"],
                sweep_params["C"],
            ),
            n_alpha=(sweep_params["electrons"] + 1) // 2,
            n_beta=sweep_params["electrons"] // 2,
            num_queries=heisenberg_queries(
                sweep_params["lambda_eff"], TARGET_PRECISION
            ),
            circuit_mapper=compiled_circuit_mapper(
                rotation_bit_precision=sweep_params["b_rot"],
                coefficient_bit_precision=sweep_params["b_coeff"],
            ),
        )
        sweep_result = estimate(
            sweep_circuit.get_qre_application(),
            architecture,
            isa_query,
            trace_query,
            max_error=MAX_ERROR,
            name=f"SOSSA {name} ({int(MEMORY_COMPUTE_PERCENTAGE * 100)}% compute)",
        )
        sweep_results.append(sweep_result)
        display(sweep_result.as_frame())

    plot_estimates(sweep_results, figsize=(8, 5))